# 训练模型

```{note}
本节我使用 Hugging Face 生态加载 tokenizer、定义模型、准备数据、训练模型并保存。做一个最小闭环，离线跑通预训练。
```

## 1. 加载 tokenizer

In [26]:
import os
from transformers import AutoTokenizer

local_model_path = os.path.expanduser(
        "~/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B/snapshots/c1899de289a04d12100db370d81485cdf75e47ca"
    )
tokenizer = AutoTokenizer.from_pretrained(local_model_path, local_files_only=True)

##

## 2. 定义模型

我们直接使用 `Qwen3ForCausalLM` 定义模型。它有如下这些可配置参数：

1. 词表和特殊 token
    * `vocab_size`：词表大小
    * `pad_token_id / bos_token_id / eos_token_id`：padding / 句首 / 句尾 token 的 id，训练时强烈建议配
2. 模型尺寸
    * `hidden_size`：每个 token 的隐藏向量维度（d_model）。
    * `intermediate_size`：MLP 中间层维度（通常约为 hidden_size 的 4～5.x 倍）。
    * `num_hidden_layers`：Transformer block 层数。
    * `initializer_range`：权重初始化标准差（normal 的 std）。
    * `rms_norm_eps`：RMSNorm 的 epsilon。
3. 注意力结构
    * `num_attention_heads`：Query 的头数。
    * `num_key_value_heads`：Key/Value 的头数，用来实现 GQA/MQA。
        * `num_key_value_heads == num_attention_heads` → 普通多头注意力 MHA
        * `num_key_value_heads == 1` → MQA
        * 其他情况 → GQA
    * `head_dim`：每个 head 的维度。
    * `attention_bias`：注意力投影（q/k/v/o）是否带 bias。
    * `attention_dropout`：注意力概率的 dropout。
4. RoPE与长上下文
    * `max_position_embeddings`：声明模型“可能用到的”最大序列长度
    * `rope_parameters`：RoPE 的参数/缩放策略容器。docstring 要求至少包含 `rope_theta`
5. Sliding Window Attention（省算力的长上下文技巧）
    * 每个 token 只看最近的 W 个历史 token（一个滑动窗口），而不是看全部历史。只有在追求长上下文训练/推理效率时才会碰它，大多数人直接默认 `use_sliding_window=False` （全 full attention）
6. 训练/推理行为开关
    * `use_cache`：生成时是否返回/使用 KV cache（加速自回归生成）。训练时通常会关掉以省显存/避免一些 Trainer 行为差异。
    * `tie_word_embeddings`：是否共享 input embedding 和输出 lm_head 权重。Qwen3 默认不共享。

In [27]:
import torch
from transformers import Qwen3Config, Qwen3ForCausalLM

# Qwen3Config 继承自 PretrainedConfig
# 这里定义一个“Qwen3-Nano”用于跑通从 0 开始的预训练流程（随机初始化 + Causal LM）
# 注意：这不是复现 Qwen3-0.6B 的规模，只是同架构缩小版。
hidden_size = 128
num_attention_heads = 2
head_dim = hidden_size // num_attention_heads
config = Qwen3Config(
    vocab_size=len(tokenizer),
    hidden_size=hidden_size,
    intermediate_size=256,
    num_hidden_layers=2,
    num_attention_heads=num_attention_heads,
    num_key_value_heads=2,
    head_dim=head_dim,
    hidden_act="silu",
    max_position_embeddings=256,
    initializer_range=0.02,
    rms_norm_eps=1e-6,
    use_cache=False,
    tie_word_embeddings=False,
    rope_parameters={"rope_type": "default", "rope_theta": 10000.0},
    pad_token_id=tokenizer.pad_token_id,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

In [28]:
# 使用随机初始化（因为是从头预训练，不是微调）
model = Qwen3ForCausalLM(config)

# 打印参数量
model_size = sum(t.numel() for t in model.parameters())
print(f"Qwen3-Nano Parameters: {model_size/1000**2:.2f}M")

Qwen3-Nano Parameters: 39.16M


## 3. 准备数据

In [29]:
from datasets import Dataset

# 这是一个演示用的 Dummy Dataset
# 实际预训练时，你应该加载大规模文本 (e.g., wikitext, c4)
# 这里的文本尽量覆盖词表里的一些 token
texts = [
    "The quick brown fox jumps over the lazy dog.",
    "To be or not to be, that is the question.",
    "I love machine learning and transformers.",
    "Qwen is a powerful language model developed by Alibaba Cloud.",
    "Hugging Face provides great tools for NLP."
] * 100 # 重复多次以构成一个 epoch

raw_dataset = Dataset.from_dict({"text": texts})
raw_dataset

Dataset({
    features: ['text'],
    num_rows: 500
})

In [30]:
# 数据处理函数
def tokenize_function(examples):
    # 超过 max_length 的 token 会被截断丢掉
    # 把每条样本都 pad 到固定长度 max_length
    # 你希望模型看到的序列长度（context length）上限为 64
    # 模型会用 attention_mask 来避免把 padding 当成有效内容
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=64)

tokenized_dataset = raw_dataset.map(tokenize_function, batched=True)
tokenized_dataset

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 500
})

In [31]:
from transformers import DataCollatorForLanguageModeling

# Data Collator: 负责把数据拼成 batch，并处理 labels
# 对于 Causal LM，labels = input_ids (shifted inside the model)
# mlm 的意思是 Masked Language Modeling，适用于 BERT/RoBERTa 类，做 Causal LM 一般不用
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

## 4. 训练

In [32]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir="./qwen3_nano_pretrain",
    max_steps=20,
    per_device_train_batch_size=2,
    save_steps=50,
    logging_steps=10,
    learning_rate=1e-4,
    weight_decay=0.01,
    bf16=True, # 如果你的 GPU 支持 BF16 (Ampere+)，建议开启
    # fp16=False,
    use_cpu=not torch.cuda.is_available(),
    report_to="none"
)

In [33]:
print("=== 开始训练 ===")
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)
trainer.train()

=== 开始训练 ===


Step,Training Loss
10,10.543797
20,10.193794


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=20, training_loss=10.368795394897461, metrics={'train_runtime': 1.7889, 'train_samples_per_second': 22.361, 'train_steps_per_second': 11.18, 'total_flos': 303240314880.0, 'train_loss': 10.368795394897461, 'epoch': 0.08})

## 5. 保存模型

In [34]:
# 保存模型和 tokenizer，使得可以 from_pretrained 加载
trainer.save_model("./qwen3_nano_final")
tokenizer.save_pretrained("./qwen3_nano_final")
print("Done! Model saved to ./qwen3_nano_final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Done! Model saved to ./qwen3_nano_final
